# Copy Local → S3  (Databricks)

Copies every file from a **source folder** (local-disk, DBFS, or Unity Catalog Volume)
to a **target S3 prefix**.  Both paths are parameterised via widgets.

### Requirements
* The cluster (or SQL warehouse) must have S3 write permission to the target prefix
  — typically via an attached **instance profile**, **service credential**, or
  **storage credential** in Unity Catalog.
* Source path examples:
    * `/Volumes/saleslake_dev/bronze_dev/vol_saleslake_src_files_dev/daily_region/`
    * `dbfs:/FileStore/uploads/region/`
    * `file:/local_disk0/tmp/region/`
* Target path examples:
    * `s3://saleslake-748005667258-eu-north-1-an/saleslake/dev/raw_files/region/`


In [0]:
# ---------- WIDGETS ----------
dbutils.widgets.text(
    name="source_path",
    defaultValue="/Volumes/saleslake_dev/bronze_dev/vol_saleslake_src_files_dev/daily_region/",
    label="Source folder (local / DBFS / Volume)",
)
dbutils.widgets.text(
    name="target_path",
    defaultValue="s3://saleslake-748005667258-eu-north-1-an/saleslake/dev/raw_files/region/",
    label="Target S3 folder",
)

source_path = dbutils.widgets.get("source_path").rstrip("/") + "/"
target_path = dbutils.widgets.get("target_path").rstrip("/") + "/"

print(f"Source : {source_path}")
print(f"Target : {target_path}")


In [0]:
# ---------- LIST FILES IN SOURCE ----------
files = [f for f in dbutils.fs.ls(source_path) if not f.isDir()]
print(f"Found {len(files)} file(s) in source:")
for f in files:
    print(f"  {f.name}  ({f.size/1024:,.1f} KB)")


In [0]:
# ---------- COPY EACH FILE TO TARGET ----------
copied = 0
failed = 0
total_bytes = 0

for f in files:
    src = f.path
    dst = target_path + f.name
    try:
        dbutils.fs.cp(src, dst)
        copied += 1
        total_bytes += f.size
        print(f"  OK   {f.name}  ->  {dst}")
    except Exception as e:
        failed += 1
        print(f"  FAIL {f.name}: {e}")

print()
print(f"Done.  copied={copied}  failed={failed}  "
      f"size={total_bytes/1024/1024:.2f} MB")


In [0]:
# ---------- VERIFY TARGET ----------
print(f"Files now at {target_path}:")
for f in dbutils.fs.ls(target_path):
    print(f"  {f.name}  ({f.size/1024:,.1f} KB)")


---
**Tip — recursive copies.**  `dbutils.fs.cp(src, dst, recurse=True)` will copy
an entire directory tree in one call. Use it instead of the per-file loop
if you don't need per-file logging:

```python
dbutils.fs.cp(source_path, target_path, recurse=True)
```